## Day_18_SQL_for_Data_People

Part A — 20 SQL queries (the deliverable)

TASK_01 Twenty queries of increasing difficulty

In [22]:
import pandas as pd
import sqlite3


# ---------------------------------------------------------
# LOAD THE THREE CSV FILES
# ---------------------------------------------------------

# Load the sales CSV into the sales DataFrame
sales = pd.read_csv("sales.csv")

# Load the customers CSV into the customers DataFrame
customers = pd.read_csv("customers.csv")

# Load the products CSV into the products DataFrame
products = pd.read_csv("products.csv")


# ---------------------------------------------------------
# CHECK THE DATA
# ---------------------------------------------------------

# Display the first 5 rows of the sales table
print("Sales table:")
display(sales.head())

# Display the first 5 rows of the customers table
print("Customers table:")
display(customers.head())

# Display the first 5 rows of the products table
print("Products table:")
display(products.head())


# ---------------------------------------------------------
# CHECK COLUMN NAMES
# ---------------------------------------------------------

# Print column names so we can confirm the correct structure
print("Sales columns:", sales.columns.tolist())
print("Customers columns:", customers.columns.tolist())
print("Products columns:", products.columns.tolist())


# ---------------------------------------------------------
# CREATE SQLITE DATABASE
# ---------------------------------------------------------

# Create an in-memory SQLite database
# The database exists while this notebook kernel is running
conn = sqlite3.connect(":memory:")


# ---------------------------------------------------------
# LOAD DATA INTO SQLITE
# ---------------------------------------------------------

# Load the sales DataFrame into the sales SQL table
sales.to_sql(
    "sales",
    conn,
    index=False,
    if_exists="replace"
)

# Load the customers DataFrame into the customers SQL table
customers.to_sql(
    "customers",
    conn,
    index=False,
    if_exists="replace"
)

# Load the products DataFrame into the products SQL table
products.to_sql(
    "products",
    conn,
    index=False,
    if_exists="replace"
)


# ---------------------------------------------------------
# CHECK SQLITE TABLES
# ---------------------------------------------------------

# Check that all three tables were successfully created
tables = pd.read_sql_query("""
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
""", conn)

display(tables)


# Q1: What are the first 10 rows of the sales table

sql = """ SELECT * FROM sales LIMIT 10;"""

result = pd.read_sql_query(sql, conn)

display(result)

# Q2: How many total orders are recorded in the sales table?

sql = """SELECT COUNT(*) AS total_orders FROM sales;"""

result = pd.read_sql_query(sql, conn)

display(result)

# Q3: Which orders have a quantity of exactly 5?

sql = """
SELECT *
FROM sales
WHERE quantity = 5;
"""

# This line executes the SQL query
result = pd.read_sql_query(sql, conn)

# Display the result
display(result)

# Q4: Which products belong to the Electronics category?

sql = """SELECT * FROM products where category = 'Electronics'; """

result = pd.read_sql_query(sql, conn)

display(result)

# Q5: What are the 5 most expensive products based on unit price?

sql = """
SELECT * FROM products 
ORDER BY unit_price DESC
LIMIT 5; """

result = pd.read_sql_query(sql, conn)

display(result)


# ----------------------------------------------------------------------

# Medium — GROUP BY / Aggregates / HAVING

# Q6: How many customers are there in each region?

sql = """
SELECT 
    region,
    COUNT(*) AS customer_count
FROM customers
GROUP BY region
ORDER BY customer_count DESC;    
"""

result = pd.read_sql_query(sql, conn)

display(result)


# Q7: What is the total quantity sold for each product?

sql = """
SELECT 
    product_id,
    SUM(quantity) AS total_quantity_sold
FROM sales
GROUP BY product_id
ORDER BY total_quantity_sold DESC;    
"""

result = pd.read_sql_query(sql, conn)

display(result)


# Q8: What is the average unit price for each product category?

sql = """ 
SELECT
    category,
    AVG(unit_price) AS average_unit_price
FROM products
GROUP BY category
ORDER BY average_unit_price DESC;    
"""  

result = pd.read_sql_query(sql, conn)

display(result)

# Q9: Which products have more than 50 orders?

sql = """ 
SELECT 
    product_id,
    COUNT(*) AS order_count
FROM sales
GROUP BY product_id
HAVING COUNT(*) > 50
ORDER BY order_count DESC;
"""

result = pd.read_sql_query(sql, conn)

display(result)

# Q10: What are the highest and lowest unit prices in the products table?

sql = """ 
SELECT 
    MAX(unit_price) AS highest_unit_price,
    MIN(unit_price) AS lowest_unit_price
FROM products;    
"""

result = pd.read_sql_query(sql, conn)

display(result)


# --------------------------------------------------------------

# Harder (JOINs):

# Q11: What is the product name and unit price for each order?

sql = """
SELECT
    s.order_id,
    s.product_id,
    p.product_name,
    p.unit_price
FROM sales AS s
JOIN products AS p
    ON s.product_id = p.product_id
ORDER BY s.order_id;
"""

# Execute the SQL query using the SQLite connection
result = pd.read_sql_query(sql, conn)

# Display the query result
display(result)

# Q12: What is the total revenue across all sales?
# Revenue = Quantity × Unit Price

sql = """
SELECT
    SUM(s.quantity * p.unit_price) AS total_revenue
FROM sales AS s
JOIN products AS p
    ON s.product_id = p.product_id;
"""

result = pd.read_sql_query(sql, conn)

display(result)

# Q13: What is the total revenue generated in each customer region?
# Region → Quantity → Product Price → Revenue

sql = """
SELECT
    c.region,
    SUM(s.quantity * p.unit_price) AS total_revenue
FROM sales AS s
JOIN customers AS c
    ON s.customer_id = c.customer_id
JOIN products AS p
    ON s.product_id = p.product_id
GROUP BY c.region
ORDER BY total_revenue DESC;
"""

result = pd.read_sql_query(sql, conn)

display(result)

# Q14: What is the total revenue generated by each product category?

sql = """
SELECT
    p.category,
    SUM(s.quantity * p.unit_price) AS total_revenue
FROM sales AS s
JOIN products AS p
    ON s.product_id = p.product_id
GROUP BY p.category
ORDER BY total_revenue DESC;
"""

result = pd.read_sql_query(sql, conn)

display(result)

# Q15: How many sales reference a customer who is missing from the customers table?

sql = """
SELECT
    COUNT(*) AS missing_customer_sales
FROM sales AS s
LEFT JOIN customers AS c
    ON s.customer_id = c.customer_id
WHERE c.customer_id IS NULL;
"""

result = pd.read_sql_query(sql, conn)

display(result)

# -------------------------------------------------------------
#  Advanced (windows / CTEs / subqueries):

# 16. Rank all products by total revenue using a window function.

sql = """
WITH product_revenue AS (
    SELECT
        p.product_id,
        p.product_name,
        SUM(s.quantity * p.unit_price) AS total_revenue
    FROM sales AS s
    JOIN products AS p
        ON s.product_id = p.product_id
    GROUP BY
        p.product_id,
        p.product_name
)

SELECT
    product_id,
    product_name,
    total_revenue,

    RANK() OVER (
        ORDER BY total_revenue DESC
    ) AS revenue_rank

FROM product_revenue

ORDER BY revenue_rank;
"""

result = pd.read_sql_query(sql, conn)

display(result)

# 17. Compute a running total of revenue by month

sql = """
WITH monthly_revenue AS (

    SELECT
        substr(date, 7, 4) || '-' || substr(date, 4, 2) AS month,

        SUM(s.quantity * p.unit_price) AS monthly_revenue

    FROM sales AS s

    JOIN products AS p
        ON s.product_id = p.product_id

    GROUP BY
        substr(date, 7, 4) || '-' || substr(date, 4, 2)
)

SELECT
    month,
    monthly_revenue,

    SUM(monthly_revenue) OVER (
        ORDER BY month
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_revenue

FROM monthly_revenue

ORDER BY month;
"""

result = pd.read_sql_query(sql, conn)

display(result)


# 18. Find the top-selling product within each category (PARTITION BY).

sql = """
WITH product_sales AS (

    SELECT
        p.category,
        p.product_id,
        p.product_name,
        SUM(s.quantity) AS total_quantity_sold

    FROM products AS p

    JOIN sales AS s
        ON p.product_id = s.product_id

    GROUP BY
        p.category,
        p.product_id,
        p.product_name
),

ranked_products AS (

    SELECT
        category,
        product_id,
        product_name,
        total_quantity_sold,

        ROW_NUMBER() OVER (
            PARTITION BY category
            ORDER BY total_quantity_sold DESC
        ) AS category_rank

    FROM product_sales
)

SELECT
    category,
    product_id,
    product_name,
    total_quantity_sold

FROM ranked_products

WHERE category_rank = 1

ORDER BY category;
"""

result = pd.read_sql_query(sql, conn)

display(result)

#  19. Using a CTE, find all products that sold above the average product revenue.

sql = """
WITH product_revenue AS (

    SELECT
        p.product_id,
        p.product_name,

        COALESCE(
            SUM(s.quantity * p.unit_price),
            0
        ) AS total_revenue

    FROM products AS p

    LEFT JOIN sales AS s
        ON p.product_id = s.product_id

    GROUP BY
        p.product_id,
        p.product_name
),

average_revenue AS (

    SELECT
        AVG(total_revenue) AS avg_product_revenue

    FROM product_revenue
)

SELECT
    pr.product_id,
    pr.product_name,
    pr.total_revenue

FROM product_revenue AS pr

CROSS JOIN average_revenue AS ar

WHERE pr.total_revenue > ar.avg_product_revenue

ORDER BY pr.total_revenue DESC;
"""

result = pd.read_sql_query(sql, conn)

display(result)


#  20. Compute each region's revenue as a percentage of total revenue.
# Region Revenue % = Region Revenue / Total Revenue × 100


sql = """
WITH region_revenue AS (

    SELECT
        c.region,

        SUM(s.quantity * p.unit_price) AS total_revenue

    FROM sales AS s

    JOIN customers AS c
        ON s.customer_id = c.customer_id

    JOIN products AS p
        ON s.product_id = p.product_id

    GROUP BY c.region
)

SELECT
    region,
    total_revenue,

    ROUND(
        100.0 * total_revenue
        / SUM(total_revenue) OVER (),
        2
    ) AS revenue_percentage

FROM region_revenue

ORDER BY revenue_percentage DESC;
"""

result = pd.read_sql_query(sql, conn)

display(result)


Sales table:


,order_id,date,customer_id,product_id,quantity
0,O00001,31-10-2024,C999,P019,2
1,O00002,14-03-2024,C999,P014,1
2,O00003,21-10-2024,C999,P004,1
3,O00004,03-01-2024,C999,P003,3
4,O00005,18-10-2024,C999,P018,4


Customers table:


,customer_id,customer_name,region,signup_year
0,C001,Customer_1,North,2024
1,C002,Customer_2,West,2021
2,C003,Customer_3,East,2022
3,C004,Customer_4,South,2022
4,C005,Customer_5,South,2022


Products table:


...

,product_id,product_name,total_revenue,revenue_rank
0,P001,Laptop,100800,1
1,P010,Standing Desk,68800,2
2,P020,Tablet,54600,3
3,P004,Monitor,44250,4
4,P009,Chair,28080,5
5,P019,Microphone,18590,6
6,P006,Headphones,17400,7
7,P018,Speaker,11520,8
8,P005,Webcam,10260,9
9,P017,Power Bank,8050,10


,month,monthly_revenue,running_revenue
0,2024-01,32561,32561
1,2024-02,47995,80556
2,2024-03,38608,119164
3,2024-04,28702,147866
4,2024-05,27291,175157
5,2024-06,31720,206877
6,2024-07,34522,241399
7,2024-08,31247,272646
8,2024-09,29860,302506
9,2024-10,30620,333126


,category,product_id,product_name,total_quantity_sold
0,Accessories,P015,Phone Stand,137
1,Electronics,P004,Monitor,177
2,Office,P008,Desk Lamp,178
3,Stationery,P011,Notebook,158


,product_id,product_name,total_revenue
0,P001,Laptop,100800
1,P010,Standing Desk,68800
2,P020,Tablet,54600
3,P004,Monitor,44250
4,P009,Chair,28080


,region,total_revenue,revenue_percentage
0,East,121636,30.10
1,West,117271,29.02
2,North,83444,20.65
3,South,81814,20.24


### Short and meaningful answers

1. **Q20 — Grand total:**
   Use a **window function with no `PARTITION BY`**, such as `SUM(total_revenue) OVER ()`. This calculates the grand total across all rows. 

2. **Easy queries:**
   Q2, Q3, Q6, Q7, Q8, and Q10 were easier because they were similar to the **Pandas operations** practiced yesterday, such as `count()`, filtering, `groupby()`, `sum()`, and `mean()`.

3. **Hardest query:**
   **Q20 was the hardest** because it required combining regional revenue with the overall grand total to calculate each region's percentage of total revenue.

**Done:** All 20 SQL queries are solved, each has a one-line plain-English comment, and each produces a result.




Part B — the same analysis in Pandas AND SQL (the deliverable)

TASK 02 Side-by-side: Pandas vs SQL

In [26]:
# ============================================================
# TASK 02: Side-by-side Pandas vs SQL
# Analysis: Total revenue per region per month
# ============================================================

import pandas as pd
import sqlite3

# ------------------------------------------------------------
# 1. Load the three CSV files
# ------------------------------------------------------------
sales = pd.read_csv("sales.csv")
customers = pd.read_csv("customers.csv")
products = pd.read_csv("products.csv")

# ------------------------------------------------------------
# 2. Create SQLite database and load the tables
# ------------------------------------------------------------
conn = sqlite3.connect(":memory:")

sales.to_sql("sales", conn, index=False, if_exists="replace")
customers.to_sql("customers", conn, index=False, if_exists="replace")
products.to_sql("products", conn, index=False, if_exists="replace")

# ============================================================
# PART A: Solve the analysis using PANDAS
# ============================================================

# Merge sales with customers to get the region

full = sales.merge(
    customers,
    on="customer_id",
    how="left"
)

# Merge the result with products to get the unit price
full = full.merge(
    products,
    on="product_id",
    how="left"
)

# Calculate revenue for each sale
full["revenue"] = full["quantity"] * full["unit_price"]

# Convert the date from DD-MM-YYYY format to datetime
full["date"] = pd.to_datetime(
    full["date"],
    format="%d-%m-%Y"
)

# Create a Year-Month column for monthly analysis
full["month"] = full["date"].dt.strftime("%Y-%m")

# Calculate total revenue for each region and month
pandas_result = (
    full.groupby(["region", "month"])["revenue"]
    .sum()
    .reset_index()
    .sort_values(["month", "region"])
)

print("========== PANDAS RESULT ==========")
display(pandas_result)

# ============================================================
# PART B: Solve the EXACT SAME analysis using SQL
# ============================================================

# SQL joins sales with customers and products,
# then groups revenue by region and month.
sql = """
SELECT
    c.region,

    -- Convert DD-MM-YYYY into YYYY-MM
    substr(s.date, 7, 4) || '-' || substr(s.date, 4, 2) AS month,

    -- Revenue = quantity × unit price
    SUM(s.quantity * p.unit_price) AS revenue

FROM sales AS s

-- Get customer region
JOIN customers AS c
    ON s.customer_id = c.customer_id

-- Get product unit price
JOIN products AS p
    ON s.product_id = p.product_id

-- Group revenue by region and month
GROUP BY
    c.region,
    month

-- Display results in month and region order
ORDER BY
    month,
    c.region;
"""

# Execute the SQL query
sql_result = pd.read_sql_query(sql, conn)

print("\n========== SQL RESULT ==========")
display(sql_result)


# ============================================================
# PART C: Compare Pandas and SQL results
# ============================================================

# Make copies so the original results remain unchanged
pandas_check = pandas_result.copy()
sql_check = sql_result.copy()

# Make sure both results have the same column names
pandas_check.columns = ["region", "month", "revenue"]
sql_check.columns = ["region", "month", "revenue"]

# Round revenue to 2 decimal places to avoid
# small floating-point differences
pandas_check["revenue"] = pandas_check["revenue"].round(2)
sql_check["revenue"] = sql_check["revenue"].round(2)

# Reset index so both DataFrames have identical indexes
pandas_check = pandas_check.reset_index(drop=True)
sql_check = sql_check.reset_index(drop=True)

# Compare both results
results_match = pandas_check.equals(sql_check)

print("\n========== COMPARISON ==========")

if results_match:
    print("Pandas and SQL results MATCH.")
else:
    print("Pandas and SQL results DO NOT MATCH.")

# Display both results side-by-side for verification
comparison = pandas_check.merge(
    sql_check,
    on=["region", "month"],
    how="outer",
    suffixes=("_pandas", "_sql")
)

print("\n========== SIDE-BY-SIDE COMPARISON ==========")
display(comparison)

========== PANDAS RESULT ==========


,region,month,revenue
0,East,2024-01,11506
12,North,2024-01,2975
24,South,2024-01,6114
36,West,2024-01,11831
1,East,2024-02,11110
13,North,2024-02,12119
25,South,2024-02,12525
37,West,2024-02,12241
2,East,2024-03,13911
14,North,2024-03,7269



========== SQL RESULT ==========


,region,month,revenue
0,East,2024-01,11506
1,North,2024-01,2975
2,South,2024-01,6114
3,West,2024-01,11831
4,East,2024-02,11110
5,North,2024-02,12119
6,South,2024-02,12525
7,West,2024-02,12241
8,East,2024-03,13911
9,North,2024-03,7269



========== COMPARISON ==========
Pandas and SQL results MATCH.

========== SIDE-BY-SIDE COMPARISON ==========


,region,month,revenue_pandas,revenue_sql
0,East,2024-01,11506,11506
1,East,2024-02,11110,11110
2,East,2024-03,13911,13911
3,East,2024-04,10275,10275
4,East,2024-05,5976,5976
5,East,2024-06,6378,6378
6,East,2024-07,9568,9568
7,East,2024-08,12035,12035
8,East,2024-09,12640,12640
9,East,2024-10,8635,8635


Pandas vs SQL Comparison : 
Pandas was easier to write because merging, calculating, and grouping data is straightforward. SQL was easier to read because the joins and aggregation are written in one query. Both methods produced matching results. I would use Pandas for analysis and visualization in Python, while I would use SQL when working directly with database tables.


### Short and meaningful answers

1. **For this question:** Pandas felt more natural because merging, creating revenue, and grouping by region and month are straightforward in Pandas.

2. **For 500 million rows:** I would use **SQL** because databases are designed to handle very large datasets efficiently and can filter and aggregate data before sending results to Python.

3. **Which is easier when?** SQL is often easier for **joins, filtering, and aggregation**, while Pandas is often easier for **data cleaning, exploratory analysis, and Python-based calculations**.

**Done:** The same analysis was completed in both Pandas and SQL, the results matched, and the approaches were compared.


Part C — get comfortable with window functions

TASK_03_Window functions practice

In [27]:
# ============================================================
# TASK 03: Window Functions Practice
# ============================================================

import pandas as pd
import sqlite3

# ------------------------------------------------------------
# 1. Load the CSV files
# ------------------------------------------------------------
sales = pd.read_csv("sales.csv")
customers = pd.read_csv("customers.csv")
products = pd.read_csv("products.csv")

# ------------------------------------------------------------
# 2. Create SQLite database and load the tables
# ------------------------------------------------------------
conn = sqlite3.connect(":memory:")

sales.to_sql("sales", conn, index=False, if_exists="replace")
customers.to_sql("customers", conn, index=False, if_exists="replace")
products.to_sql("products", conn, index=False, if_exists="replace")


# ============================================================
# Q1. Running total of revenue over the whole year
# ============================================================

# First calculate revenue for each date,
# then use SUM() OVER() to calculate the cumulative total.
sql = """
WITH daily_revenue AS (
    SELECT
        s.date,
        SUM(s.quantity * p.unit_price) AS revenue
    FROM sales AS s
    JOIN products AS p
        ON s.product_id = p.product_id
    GROUP BY s.date
)

SELECT
    date,
    revenue,
    SUM(revenue) OVER (
        ORDER BY date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_total
FROM daily_revenue
ORDER BY date;
"""

result = pd.read_sql_query(sql, conn)

print("Q1: Running Total of Revenue")
display(result)


# ============================================================
# Q2. Rank customers by their total spending
# ============================================================

# Calculate each customer's total spending,
# then rank customers from highest to lowest spending.
sql = """
WITH customer_spending AS (
    SELECT
        s.customer_id,
        c.customer_name,
        SUM(s.quantity * p.unit_price) AS total_spending
    FROM sales AS s
    JOIN customers AS c
        ON s.customer_id = c.customer_id
    JOIN products AS p
        ON s.product_id = p.product_id
    GROUP BY
        s.customer_id,
        c.customer_name
)

SELECT
    customer_id,
    customer_name,
    total_spending,
    RANK() OVER (
        ORDER BY total_spending DESC
    ) AS spending_rank
FROM customer_spending
ORDER BY spending_rank
LIMIT 10;
"""

result = pd.read_sql_query(sql, conn)

print("\nQ2: Top 10 Customers by Total Spending")
display(result)


# ============================================================
# Q3. Rank customers within each region
# ============================================================

# PARTITION BY region restarts the ranking for every region.
sql = """
WITH customer_spending AS (
    SELECT
        c.region,
        s.customer_id,
        c.customer_name,
        SUM(s.quantity * p.unit_price) AS total_spending
    FROM sales AS s
    JOIN customers AS c
        ON s.customer_id = c.customer_id
    JOIN products AS p
        ON s.product_id = p.product_id
    GROUP BY
        c.region,
        s.customer_id,
        c.customer_name
)

SELECT
    region,
    customer_id,
    customer_name,
    total_spending,
    RANK() OVER (
        PARTITION BY region
        ORDER BY total_spending DESC
    ) AS regional_rank
FROM customer_spending
ORDER BY
    region,
    regional_rank;
"""

result = pd.read_sql_query(sql, conn)

print("\nQ3: Customer Ranking Within Each Region")
display(result)


# ============================================================
# Q4. Monthly revenue and previous month's revenue
# ============================================================

# LAG() gets the revenue from the previous month.
sql = """
WITH monthly_revenue AS (
    SELECT
        substr(s.date, 7, 4) || '-' || substr(s.date, 4, 2) AS month,
        SUM(s.quantity * p.unit_price) AS revenue
    FROM sales AS s
    JOIN products AS p
        ON s.product_id = p.product_id
    GROUP BY month
)

SELECT
    month,
    revenue,
    LAG(revenue) OVER (
        ORDER BY month
    ) AS previous_month_revenue
FROM monthly_revenue
ORDER BY month;
"""

result = pd.read_sql_query(sql, conn)

print("\nQ4: Monthly Revenue vs Previous Month")
display(result)


# ============================================================
# Q5. Month-over-month revenue growth
# ============================================================

# Calculate the percentage change from the previous month.
# Formula:
# ((Current Revenue - Previous Revenue) / Previous Revenue) * 100
sql = """
WITH monthly_revenue AS (
    SELECT
        substr(s.date, 7, 4) || '-' || substr(s.date, 4, 2) AS month,
        SUM(s.quantity * p.unit_price) AS revenue
    FROM sales AS s
    JOIN products AS p
        ON s.product_id = p.product_id
    GROUP BY month
),

monthly_with_previous AS (
    SELECT
        month,
        revenue,
        LAG(revenue) OVER (
            ORDER BY month
        ) AS previous_month_revenue
    FROM monthly_revenue
)

SELECT
    month,
    revenue,
    previous_month_revenue,

    CASE
        WHEN previous_month_revenue IS NULL
             OR previous_month_revenue = 0
        THEN NULL
        ELSE ROUND(
            ((revenue - previous_month_revenue)
            * 100.0 / previous_month_revenue), 2
        )
    END AS mom_growth_percent

FROM monthly_with_previous
ORDER BY month;
"""

result = pd.read_sql_query(sql, conn)

print("\nQ5: Month-over-Month Revenue Growth")
display(result)

Q1: Running Total of Revenue


,date,revenue,running_total
0,01-02-2024,4075,4075
1,01-03-2024,617,4692
2,01-04-2024,1405,6097
3,01-05-2024,36,6133
4,01-06-2024,1740,7873
...,...,...,...
338,31-05-2024,320,397494
339,31-07-2024,916,398410
340,31-08-2024,980,399390
341,31-10-2024,1670,401060



Q2: Top 10 Customers by Total Spending


,customer_id,customer_name,total_spending,spending_rank
0,C024,Customer_24,19300,1
1,C015,Customer_15,17480,2
2,C003,Customer_3,12885,3
3,C039,Customer_39,12568,4
4,C023,Customer_23,12464,5
5,C043,Customer_43,12214,6
6,C027,Customer_27,11775,7
7,C050,Customer_50,11644,8
8,C007,Customer_7,11114,9
9,C029,Customer_29,10887,10



Q3: Customer Ranking Within Each Region


,region,customer_id,customer_name,total_spending,regional_rank
0,East,C015,Customer_15,17480,1
1,East,C003,Customer_3,12885,2
2,East,C043,Customer_43,12214,3
3,East,C029,Customer_29,10887,4
4,East,C034,Customer_34,10843,5
5,East,C049,Customer_49,9920,6
6,East,C021,Customer_21,9745,7
7,East,C008,Customer_8,8986,8
8,East,C017,Customer_17,5967,9
9,East,C011,Customer_11,5905,10



Q4: Monthly Revenue vs Previous Month


,month,revenue,previous_month_revenue
0,2024-01,32561,NaN
1,2024-02,47995,32561.0
2,2024-03,38608,47995.0
3,2024-04,28702,38608.0
4,2024-05,27291,28702.0
5,2024-06,31720,27291.0
6,2024-07,34522,31720.0
7,2024-08,31247,34522.0
8,2024-09,29860,31247.0
9,2024-10,30620,29860.0



Q5: Month-over-Month Revenue Growth


,month,revenue,previous_month_revenue,mom_growth_percent
0,2024-01,32561,NaN,NaN
1,2024-02,47995,32561.0,47.40
2,2024-03,38608,47995.0,-19.56
3,2024-04,28702,38608.0,-25.66
4,2024-05,27291,28702.0,-4.92
5,2024-06,31720,27291.0,16.23
6,2024-07,34522,31720.0,8.83
7,2024-08,31247,34522.0,-9.49
8,2024-09,29860,31247.0,-4.44
9,2024-10,30620,29860.0,2.55


### Short and meaningful answers

1. **RANK vs DENSE_RANK vs ROW_NUMBER:**
   `RANK()` gives the same rank to ties but leaves gaps. `DENSE_RANK()` gives the same rank to ties without gaps. `ROW_NUMBER()` gives every row a unique number. This matters when deciding how ties should be handled. \

2. **LAG vs LEAD:**
   `LAG()` looks at a previous row, while `LEAD()` looks at the next row. They are useful for comparing values such as this month's revenue with the previous or next month. \

3. **Why window functions can be cleaner:**
   Window functions can calculate rankings, running totals, and previous-row values directly in SQL without repeatedly creating separate grouped DataFrames. This can make database-based analysis more concise. ([SQLite][2])

**Done:** Running total, customer ranking, regional ranking with `PARTITION BY`, and month-over-month growth using `LAG()` were completed.
\
